# Step 1 — prepare inputs for the block-merging workflow

**# of cells in notebook:** 6 code cells

## Purpose

Prepare the block and building datasets required for the block-merging workflow. The notebook creates the working `segments_v2` workspace, checks and repairs block geometry, assigns each block to a `zones_5` zone, selects the Overture buildings associated with the analysis blocks, creates block-level exclusion and geometry fields, and exports the finalized `blocks_5` layer used by the subsequent notebooks.

## Input

- Blocks layer in a geodatabase
- `zones_5` layer in a geodatabase
- Overture buildings layer in a GeoPackage

**Johannesburg blocks**
- Geodatabase: `E:\_johannesburg\_analysis\blocks\blocks.gdb`
- Layer: `johannesburg_blocks_utm35s_hetero_swapped_v2_SPARC`

**Johannesburg zones**
- Geodatabase: `E:\_johannesburg\_analysis\zones\zones.gdb`
- Layer: `zones_5`
- Required field: `population`

**Overture building footprints**
- GeoPackage: `E:\_johannesburg\ZAF_johannesburg_overture_buildings_20260610.gpkg`
- Layer: `overture_buildings_clipped_to_blocks`

## Output

- `segments_v2` workspace with `segments.gdb`
- `blocks_5` geodatabase layer
- Buildings selection + projection
- `blocks_5` GeoPackage layer

**Working geodatabase**
- `E:\_johannesburg\_analysis\segments_v2\segments.gdb`

**Prepared block layer**
- `segments.gdb\blocks_5`

**Selected Overture buildings**
- `E:\_johannesburg\_analysis\buildings\buildings.gdb\building_centroid_in_blocks`

**Projected selected buildings**
- `E:\_johannesburg\_analysis\buildings\buildings.gdb\building_centroid_in_blocks_utm35s`

**Final block GeoPackage**
- `E:\_johannesburg\_analysis\segments_v2\blocks_5.gpkg`
- Layer: `blocks_5`

The final `blocks_5` GeoPackage layer retains the core source attributes together with:

- `zones_5_ID`
- `zones_5_pop`
- `open_space`
- `airport`
- `block_area_m2`
- `block_perimeter_m`

## Main logic

### Cell 1 — create the block-merging workspace

1. Define `E:\_johannesburg\_analysis` as the analysis directory.
2. Create the `segments_v2` folder if it does not already exist.
3. Create `segments.gdb` inside `segments_v2` if it does not already exist.

### Cell 2 — check source block and zone geometry

1. Run ArcGIS `CheckGeometry` on the input blocks.
2. Run the same check on `zones_5`.
3. Write geometry-check tables to `segments.gdb`.
4. Report the number of geometry problems found and, when problems exist, print the first affected feature IDs and problem types.

This cell is diagnostic only and does not modify the source datasets.

### Cell 3 — create `blocks_5` and assign zones

1. Copy the source blocks to a temporary working feature class.
2. Check block geometry before repair.
3. Run `RepairGeometry` on the working copy rather than the source blocks.
4. Check geometry again after repair and report:
   - geometry problems before and after repair;
   - whether the number of block features changed during repair.
5. Create the final geodatabase layer `blocks_5` from the repaired working copy.
6. Add:
   - `zones_5_ID`
   - `zones_5_pop`
7. Create a temporary copy of the zone geometry while explicitly preserving:
   - the original `OBJECTID` from `zones_5`;
   - the zone population.
8. Create one guaranteed-inside point for each block.
9. Spatially join the block inside-points to `zones_5`.
10. Require every block point to match exactly one zone:
    - `Join_Count = 1` is valid;
    - `Join_Count = 0` causes the workflow to stop;
    - `Join_Count > 1` also causes the workflow to stop.
11. Transfer the original `zones_5` `OBJECTID` to `zones_5_ID` and the zone population to `zones_5_pop`.

Thus, `zones_5_ID` refers to the `OBJECTID` of the original `zones_5` layer rather than the ObjectID of a temporary copy.

### Cell 4 — select and project Overture buildings

1. Confirm that `blocks_5` exists and uses a projected coordinate system.
2. Create the `buildings` folder and `buildings.gdb` if needed.
3. Use ArcGIS `SelectLayerByLocation` to select Overture building polygons using:
   - relationship: `HAVE_THEIR_CENTER_IN`;
   - selecting features: `blocks_5`;
   - selection type: `NEW_SELECTION`.
4. Export the selected building polygons as `building_centroid_in_blocks`.
5. Project the selected buildings to WGS 84 / UTM Zone 35S (`EPSG:32735`).
6. Save the projected layer as `building_centroid_in_blocks_utm35s`.

Despite the layer name, these outputs remain building polygons; the center criterion is used to determine which buildings are selected.

### Cell 5 — create block flags and geometry attributes

1. Add four fields to `blocks_5` if they do not already exist:
   - `open_space` — short integer;
   - `airport` — short integer;
   - `block_area_m2` — double;
   - `block_perimeter_m` — double.
2. Calculate `open_space`:
   - `1` where `composite_class = 'open_space'`;
   - `0` for all other blocks.
3. Calculate `airport`:
   - `1` where `composite_class = 'airport'`;
   - `0` for all other blocks.
4. Calculate block area in square meters.
5. Calculate block perimeter in meters.

The `open_space` and `airport` fields are later used to identify blocks that must remain standalone rather than participate in block merging.

### Cell 6 — export the finalized `blocks_5` layer

1. Confirm that all required output fields exist.
2. Create a new `blocks_5.gpkg`.
3. Export the prepared block polygons while retaining only the selected attributes required by the downstream workflow.
4. Confirm the number of features written to the output.

The resulting `blocks_5.gpkg` is the principal block input to the subsequent building-statistics, topology-diagnostics, and block-merging steps.


In [ ]:
import arcpy
import os

analysis_folder = r"E:\_johannesburg\_analysis"
segments_folder = os.path.join(analysis_folder, "segments_v2")
segments_gdb = os.path.join(segments_folder, "segments.gdb")

os.makedirs(segments_folder, exist_ok=True)

if not arcpy.Exists(segments_gdb):
    arcpy.management.CreateFileGDB(segments_folder, "segments.gdb")

print(segments_gdb)

In [ ]:
import arcpy
import os

blocks_fc = r"E:\_johannesburg\_analysis\blocks\blocks.gdb\johannesburg_blocks_utm35s_hetero_swapped_v2_SPARC"
zones_fc = r"E:\_johannesburg\_analysis\zones\zones.gdb\zones_5"
out_gdb = r"E:\_johannesburg\_analysis\segments_v2\segments.gdb"

def check_geometry(fc, name):
    out_table = os.path.join(out_gdb, f"check_geometry_{name}")

    if arcpy.Exists(out_table):
        arcpy.management.Delete(out_table)

    print(f"Checking geometry: {name}")
    arcpy.management.CheckGeometry(fc, out_table)

    n = int(arcpy.management.GetCount(out_table)[0])
    print(f"Geometry problems found: {n:,}")

    if n > 0:
        fields = [f.name for f in arcpy.ListFields(out_table)]
        print("Fields in check table:")
        print(fields)

        show_fields = [f for f in ["FEATURE_ID", "PROBLEM"] if f in fields]

        if show_fields:
            print("\nFirst geometry problems:")
            with arcpy.da.SearchCursor(out_table, show_fields) as cur:
                for i, row in enumerate(cur):
                    print(row)
                    if i >= 20:
                        break

    print("")

check_geometry(blocks_fc, "blocks")
check_geometry(zones_fc, "zones")

In [ ]:
import arcpy
import os
import time
import traceback

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

blocks_fc = (
    r"E:\_johannesburg\_analysis\blocks\blocks.gdb"
    r"\johannesburg_blocks_utm35s_hetero_swapped_v2_SPARC"
)

zones_fc = (
    r"E:\_johannesburg\_analysis\zones\zones.gdb"
    r"\zones_5"
)

out_gdb = r"E:\_johannesburg\_analysis\segments_v2\segments.gdb"

out_fc_name = "blocks_5"
out_fc = os.path.join(out_gdb, out_fc_name)

zone_pop_field = "population"

out_zone_id_field = "zones_5_ID"
out_zone_pop_field = "zones_5_pop"

overwrite_output = True

# ------------------------------------------------------------
# Temporary working datasets
# ------------------------------------------------------------

work_blocks = os.path.join(
    out_gdb,
    "_work_blocks_repaired"
)

work_zones = os.path.join(
    out_gdb,
    "_work_zones_for_join"
)

block_points = os.path.join(
    out_gdb,
    "_work_block_inside_points"
)

block_points_joined = os.path.join(
    out_gdb,
    "_work_block_inside_points_joined"
)

check_before = os.path.join(
    out_gdb,
    "_check_blocks_before_repair"
)

check_after = os.path.join(
    out_gdb,
    "_check_blocks_after_repair"
)

# These are deliberately created as ordinary fields in work_zones.
# join_z5_id will contain the ORIGINAL zones_5 OBJECTID.
join_zone_id_field = "join_z5_id"
join_zone_pop_field = "join_z5_pop"


# ------------------------------------------------------------
# ArcPy setup
# ------------------------------------------------------------

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def log(msg):
    print(msg, flush=True)


def field_exists(fc, field_name):
    return any(
        f.name.lower() == field_name.lower()
        for f in arcpy.ListFields(fc)
    )


def get_actual_field_name(fc, field_name):
    for field in arcpy.ListFields(fc):
        if field.name.lower() == field_name.lower():
            return field.name

    return None


def add_field_if_missing(
    fc,
    field_name,
    field_type
):
    if not field_exists(fc, field_name):
        arcpy.management.AddField(
            fc,
            field_name,
            field_type
        )

        log(f"Added field: {field_name}")

    else:
        log(
            f"Field already exists: "
            f"{field_name}"
        )


def delete_if_exists(path):
    if arcpy.Exists(path):
        log(
            f"Deleting existing:\n"
            f"  {path}"
        )

        arcpy.management.Delete(path)


def count_rows(path):
    return int(
        arcpy.management.GetCount(path)[0]
    )


def get_oid_field(fc):
    return arcpy.Describe(fc).OIDFieldName


def check_geometry(
    fc,
    out_table,
    label
):
    delete_if_exists(out_table)

    log(
        f"Checking geometry: "
        f"{label}"
    )

    arcpy.management.CheckGeometry(
        fc,
        out_table
    )

    n = count_rows(out_table)

    log(
        f"  Geometry problems: "
        f"{n:,}"
    )

    if n > 0:

        fields = [
            f.name
            for f in arcpy.ListFields(out_table)
        ]

        show_fields = [
            f
            for f in [
                "FEATURE_ID",
                "PROBLEM"
            ]
            if f in fields
        ]

        if show_fields:

            log(
                "  First few problems:"
            )

            with arcpy.da.SearchCursor(
                out_table,
                show_fields
            ) as cur:

                for i, row in enumerate(cur):

                    log(
                        f"    {row}"
                    )

                    if i >= 10:
                        break

    return n


def create_work_zones():
    """
    Create a stripped-down working zones layer containing:

        geometry
        join_z5_id  = ORIGINAL zones_5 OBJECTID
        join_z5_pop = population

    This avoids relying on the ObjectID created by CopyFeatures.
    """

    delete_if_exists(work_zones)

    desc = arcpy.Describe(zones_fc)

    source_oid_field = desc.OIDFieldName
    source_sr = desc.spatialReference
    shape_type = desc.shapeType

    population_field_actual = get_actual_field_name(
        zones_fc,
        zone_pop_field
    )

    if population_field_actual is None:
        raise ValueError(
            f"Zones layer is missing field: "
            f"{zone_pop_field}"
        )

    log(
        "Creating working zones layer while "
        "preserving original zones_5 OBJECTIDs..."
    )

    arcpy.management.CreateFeatureclass(
        out_path=out_gdb,
        out_name=os.path.basename(work_zones),
        geometry_type=shape_type,
        spatial_reference=source_sr
    )

    arcpy.management.AddField(
        work_zones,
        join_zone_id_field,
        "LONG"
    )

    arcpy.management.AddField(
        work_zones,
        join_zone_pop_field,
        "DOUBLE"
    )

    read_fields = [
        "SHAPE@",
        source_oid_field,
        population_field_actual
    ]

    write_fields = [
        "SHAPE@",
        join_zone_id_field,
        join_zone_pop_field
    ]

    n_written = 0

    with arcpy.da.SearchCursor(
        zones_fc,
        read_fields
    ) as search_cur:

        with arcpy.da.InsertCursor(
            work_zones,
            write_fields
        ) as insert_cur:

            for (
                geometry,
                source_oid,
                zone_pop
            ) in search_cur:

                insert_cur.insertRow([
                    geometry,
                    source_oid,
                    zone_pop
                ])

                n_written += 1

    log(
        f"Working zones created: "
        f"{n_written:,}"
    )

    log(
        f"{join_zone_id_field} contains "
        f"the original {source_oid_field} "
        f"from zones_5."
    )


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():

    t0 = time.time()

    # --------------------------------------------------------
    # Check inputs
    # --------------------------------------------------------

    log("=" * 80)
    log("CHECKING INPUTS")
    log("=" * 80)

    if not arcpy.Exists(blocks_fc):
        raise FileNotFoundError(
            f"Blocks layer does not exist:\n"
            f"{blocks_fc}"
        )

    if not arcpy.Exists(zones_fc):
        raise FileNotFoundError(
            f"Zones layer does not exist:\n"
            f"{zones_fc}"
        )

    if not arcpy.Exists(out_gdb):

        log(
            f"Creating output geodatabase:\n"
            f"{out_gdb}"
        )

        parent_folder = os.path.dirname(
            out_gdb
        )

        gdb_name = os.path.basename(
            out_gdb
        )

        arcpy.management.CreateFileGDB(
            parent_folder,
            gdb_name
        )

    if not field_exists(
        zones_fc,
        zone_pop_field
    ):
        raise ValueError(
            f"Zones layer is missing field: "
            f"{zone_pop_field}"
        )

    log(
        f"Blocks: {blocks_fc}"
    )

    log(
        f"Zones:  {zones_fc}"
    )

    log(
        f"Original zones OID field: "
        f"{get_oid_field(zones_fc)}"
    )

    # --------------------------------------------------------
    # Clean old temporary datasets
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CLEANING OLD WORKING DATA")
    log("=" * 80)

    for path in [
        work_blocks,
        work_zones,
        block_points,
        block_points_joined,
        check_before,
        check_after,
    ]:
        delete_if_exists(path)

    if arcpy.Exists(out_fc):

        if overwrite_output:
            delete_if_exists(out_fc)

        else:
            raise FileExistsError(
                f"Output already exists:\n"
                f"{out_fc}"
            )

    # --------------------------------------------------------
    # Copy blocks
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("COPYING AND REPAIRING BLOCKS")
    log("=" * 80)

    log(
        "Copying blocks to working layer..."
    )

    arcpy.management.CopyFeatures(
        blocks_fc,
        work_blocks
    )

    n_blocks_before = count_rows(
        work_blocks
    )

    log(
        f"Working block count before repair: "
        f"{n_blocks_before:,}"
    )

    # --------------------------------------------------------
    # Check geometry before repair
    # --------------------------------------------------------

    problems_before = check_geometry(
        work_blocks,
        check_before,
        "blocks before repair"
    )

    # --------------------------------------------------------
    # Repair
    # --------------------------------------------------------

    log("")
    log(
        "Repairing block geometry "
        "on working copy..."
    )

    arcpy.management.RepairGeometry(
        work_blocks
    )

    n_blocks_after = count_rows(
        work_blocks
    )

    log(
        f"Working block count after repair: "
        f"{n_blocks_after:,}"
    )

    # --------------------------------------------------------
    # Check geometry after repair
    # --------------------------------------------------------

    problems_after = check_geometry(
        work_blocks,
        check_after,
        "blocks after repair"
    )

    # --------------------------------------------------------
    # Repair diagnostics
    # --------------------------------------------------------

    log("")
    log("Repair diagnostics:")

    log(
        f"  Problems before repair: "
        f"{problems_before:,}"
    )

    log(
        f"  Problems after repair:  "
        f"{problems_after:,}"
    )

    if n_blocks_after != n_blocks_before:

        log("")
        log("WARNING:")

        log(
            "  Block count changed during "
            "RepairGeometry:"
        )

        log(
            f"  {n_blocks_before:,} "
            f"-> {n_blocks_after:,}"
        )

        log(
            "  Inspect this before using "
            "the output as final."
        )

    if problems_after > 0:

        log("")
        log("WARNING:")

        log(
            f"  {problems_after:,} geometry "
            f"problem(s) remain after repair."
        )

        log(
            "  Inspect the post-repair "
            "Check Geometry table."
        )

    # --------------------------------------------------------
    # Create final blocks_5
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CREATING BLOCKS_5")
    log("=" * 80)

    arcpy.management.CopyFeatures(
        work_blocks,
        out_fc
    )

    log(
        f"Created:\n"
        f"  {out_fc}"
    )

    add_field_if_missing(
        out_fc,
        out_zone_id_field,
        "LONG"
    )

    add_field_if_missing(
        out_fc,
        out_zone_pop_field,
        "DOUBLE"
    )

    # --------------------------------------------------------
    # Create work_zones while preserving original OBJECTID
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("PREPARING ZONES")
    log("=" * 80)

    create_work_zones()

    # --------------------------------------------------------
    # Convert blocks to inside points
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CREATING BLOCK INSIDE POINTS")
    log("=" * 80)

    arcpy.management.FeatureToPoint(
        out_fc,
        block_points,
        "INSIDE"
    )

    n_points = count_rows(
        block_points
    )

    log(
        f"Inside points created: "
        f"{n_points:,}"
    )

    # --------------------------------------------------------
    # Spatial join points to zones
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("SPATIAL JOIN")
    log("=" * 80)

    log(
        "Spatial joining block inside-points "
        "to zones..."
    )

    arcpy.analysis.SpatialJoin(
        target_features=block_points,
        join_features=work_zones,
        out_feature_class=block_points_joined,
        join_operation="JOIN_ONE_TO_ONE",
        join_type="KEEP_ALL",
        match_option="INTERSECT"
    )

    # --------------------------------------------------------
    # Resolve joined field names
    # --------------------------------------------------------

    joined_fields = {
        f.name.lower(): f.name
        for f in arcpy.ListFields(
            block_points_joined
        )
    }

    required_join_fields = [
        "ORIG_FID",
        "Join_Count",
        join_zone_id_field,
        join_zone_pop_field
    ]

    missing_join_fields = []

    for field_name in required_join_fields:

        if field_name.lower() not in joined_fields:
            missing_join_fields.append(
                field_name
            )

    if missing_join_fields:

        raise RuntimeError(
            "Expected fields were not found "
            "in spatial join output:\n"
            +
            "\n".join(
                f"  - {f}"
                for f in missing_join_fields
            )
        )

    orig_fid_actual = joined_fields[
        "orig_fid"
    ]

    join_count_actual = joined_fields[
        "join_count"
    ]

    join_id_actual = joined_fields[
        join_zone_id_field.lower()
    ]

    join_pop_actual = joined_fields[
        join_zone_pop_field.lower()
    ]

    # --------------------------------------------------------
    # Validate Join_Count
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("VALIDATING POINT-ZONE MATCHES")
    log("=" * 80)

    n_join_zero = 0
    n_join_one = 0
    n_join_multiple = 0

    offenders = []

    with arcpy.da.SearchCursor(
        block_points_joined,
        [
            orig_fid_actual,
            join_count_actual
        ]
    ) as cur:

        for orig_fid, join_count in cur:

            if join_count == 0:

                n_join_zero += 1

                if len(offenders) < 25:

                    offenders.append(
                        (
                            orig_fid,
                            join_count
                        )
                    )

            elif join_count == 1:

                n_join_one += 1

            else:

                n_join_multiple += 1

                if len(offenders) < 25:

                    offenders.append(
                        (
                            orig_fid,
                            join_count
                        )
                    )

    log(
        f"Exactly one zone match: "
        f"{n_join_one:,}"
    )

    log(
        f"Zero zone matches:        "
        f"{n_join_zero:,}"
    )

    log(
        f"Multiple zone matches:    "
        f"{n_join_multiple:,}"
    )

    if (
        n_join_zero > 0
        or
        n_join_multiple > 0
    ):

        log("")
        log(
            "First offending block "
            "inside-points:"
        )

        log(
            "  (ORIG_FID, Join_Count)"
        )

        for offender in offenders:

            log(
                f"  {offender}"
            )

        raise RuntimeError(
            "\nSpatial join validation failed.\n"
            "Every block inside-point must "
            "intersect exactly one zones_5 polygon.\n"
            f"Zero matches: {n_join_zero:,}\n"
            f"Multiple matches: "
            f"{n_join_multiple:,}\n"
            "blocks_5 zone fields were NOT updated."
        )

    log("")
    log(
        "Spatial join validation passed: "
        "every block matches exactly one zone."
    )

    # --------------------------------------------------------
    # Build lookup
    # --------------------------------------------------------

    log("")
    log(
        "Reading validated point-zone "
        "matches..."
    )

    match_lookup = {}

    with arcpy.da.SearchCursor(
        block_points_joined,
        [
            orig_fid_actual,
            join_id_actual,
            join_pop_actual
        ]
    ) as cur:

        for (
            orig_fid,
            zone_id,
            zone_pop
        ) in cur:

            match_lookup[orig_fid] = (
                zone_id,
                zone_pop
            )

    log(
        f"Point-zone lookup records: "
        f"{len(match_lookup):,}"
    )

    # --------------------------------------------------------
    # Update blocks_5
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("UPDATING BLOCK ATTRIBUTES")
    log("=" * 80)

    log(
        "Updating blocks_5 with "
        "zones_5_ID and zones_5_pop..."
    )

    out_oid_field = get_oid_field(
        out_fc
    )

    n_total = 0
    n_matched = 0
    n_unmatched = 0

    with arcpy.da.UpdateCursor(
        out_fc,
        [
            out_oid_field,
            out_zone_id_field,
            out_zone_pop_field
        ]
    ) as cur:

        for (
            oid,
            old_zone_id,
            old_zone_pop
        ) in cur:

            n_total += 1

            zone_id, zone_pop = (
                match_lookup.get(
                    oid,
                    (None, None)
                )
            )

            if zone_id is None:
                n_unmatched += 1

            else:
                n_matched += 1

            cur.updateRow([
                oid,
                zone_id,
                zone_pop
            ])

            if n_total % 1000 == 0:

                log(
                    f"  Updated "
                    f"{n_total:,} blocks..."
                )

    # This should be impossible after Join_Count validation,
    # but retain the check as a final safeguard.
    if n_unmatched > 0:

        raise RuntimeError(
            f"{n_unmatched:,} blocks were "
            f"unexpectedly unmatched during "
            f"the final update."
        )

    # --------------------------------------------------------
    # Final report
    # --------------------------------------------------------

    elapsed = time.time() - t0

    log("")
    log("=" * 80)
    log("DONE")
    log("=" * 80)

    log(
        f"Output layer:\n"
        f"  {out_fc}"
    )

    log("")
    log(
        f"Total blocks:   "
        f"{n_total:,}"
    )

    log(
        f"Matched blocks: "
        f"{n_matched:,}"
    )

    log(
        f"Unmatched:      "
        f"{n_unmatched:,}"
    )

    log("")
    log(
        f"Geometry problems before repair: "
        f"{problems_before:,}"
    )

    log(
        f"Geometry problems after repair:  "
        f"{problems_after:,}"
    )

    log("")
    log(
        f"Elapsed: "
        f"{elapsed / 60:.2f} minutes"
    )


if __name__ == "__main__":

    try:
        main()

    except Exception:

        log("")
        log("ERROR:")
        log(
            traceback.format_exc()
        )

In [ ]:
import arcpy
import os
import traceback

# ------------------------------------------------------------
# Inputs
# ------------------------------------------------------------

analysis_folder = r"E:\_johannesburg\_analysis"

segments_gdb = os.path.join(
    analysis_folder,
    "segments_v2",
    "segments.gdb"
)

blocks_fc = os.path.join(
    segments_gdb,
    "blocks_5"
)

buildings_input = (
    r"E:\_johannesburg\ZAF_johannesburg_overture_buildings_20260610.gpkg"
    r"\main.overture_buildings_clipped_to_blocks"
)

# ------------------------------------------------------------
# Output buildings folder / geodatabase
# ------------------------------------------------------------

buildings_folder = os.path.join(
    analysis_folder,
    "buildings"
)

buildings_gdb = os.path.join(
    buildings_folder,
    "buildings.gdb"
)

selected_buildings = os.path.join(
    buildings_gdb,
    "building_centroid_in_blocks"
)

projected_buildings = os.path.join(
    buildings_gdb,
    "building_centroid_in_blocks_utm35s"
)

# Johannesburg = WGS 1984 UTM Zone 35S
target_sr = arcpy.SpatialReference(32735)

# ------------------------------------------------------------
# ArcPy settings
# ------------------------------------------------------------

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def log(msg):
    print(msg, flush=True)


def delete_if_exists(path):
    if arcpy.Exists(path):
        log(f"Deleting existing output:\n  {path}")
        arcpy.management.Delete(path)


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():

    # --------------------------------------------------------
    # 1. Check blocks_5
    # --------------------------------------------------------

    log("=" * 80)
    log("CHECKING BLOCKS_5")
    log("=" * 80)

    if not arcpy.Exists(blocks_fc):
        raise FileNotFoundError(
            f"blocks_5 does not exist:\n{blocks_fc}"
        )

    blocks_desc = arcpy.Describe(blocks_fc)
    blocks_sr = blocks_desc.spatialReference

    if blocks_sr is None:
        raise ValueError(
            "blocks_5 does not have a readable spatial reference."
        )

    if blocks_sr.type != "Projected":
        raise ValueError(
            "blocks_5 is not in a projected coordinate system.\n"
            f"Current coordinate system: {blocks_sr.name}"
        )

    log(f"blocks_5 exists: {blocks_fc}")
    log(f"Coordinate system: {blocks_sr.name}")
    log(f"Coordinate system type: {blocks_sr.type}")
    log(f"WKID: {blocks_sr.factoryCode}")

    block_count = int(arcpy.management.GetCount(blocks_fc)[0])
    log(f"Block count: {block_count:,}")

    # --------------------------------------------------------
    # 2. Check building input
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CHECKING BUILDING INPUT")
    log("=" * 80)

    if not arcpy.Exists(buildings_input):
        raise FileNotFoundError(
            f"Building input does not exist:\n{buildings_input}"
        )

    building_count = int(
        arcpy.management.GetCount(buildings_input)[0]
    )

    building_sr = arcpy.Describe(
        buildings_input
    ).spatialReference

    log(f"Building input exists: {buildings_input}")
    log(f"Building count: {building_count:,}")
    log(f"Building CRS: {building_sr.name}")

    # --------------------------------------------------------
    # 3. Create buildings folder
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CREATING BUILDINGS WORKSPACE")
    log("=" * 80)

    if not os.path.isdir(buildings_folder):
        os.makedirs(buildings_folder)
        log(f"Created folder:\n  {buildings_folder}")
    else:
        log(f"Folder already exists:\n  {buildings_folder}")

    # --------------------------------------------------------
    # 4. Create buildings.gdb
    # --------------------------------------------------------

    if not arcpy.Exists(buildings_gdb):
        arcpy.management.CreateFileGDB(
            out_folder_path=buildings_folder,
            out_name="buildings.gdb"
        )
        log(f"Created geodatabase:\n  {buildings_gdb}")
    else:
        log(f"Geodatabase already exists:\n  {buildings_gdb}")

    # --------------------------------------------------------
    # 5. Remove previous outputs if present
    # --------------------------------------------------------

    delete_if_exists(selected_buildings)
    delete_if_exists(projected_buildings)

    # --------------------------------------------------------
    # 6. Create feature layers
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("SELECTING BUILDINGS")
    log("=" * 80)

    buildings_lyr = "overture_buildings_selection_lyr"
    blocks_lyr = "blocks_5_selection_lyr"

    if arcpy.Exists(buildings_lyr):
        arcpy.management.Delete(buildings_lyr)

    if arcpy.Exists(blocks_lyr):
        arcpy.management.Delete(blocks_lyr)

    arcpy.management.MakeFeatureLayer(
        buildings_input,
        buildings_lyr
    )

    arcpy.management.MakeFeatureLayer(
        blocks_fc,
        blocks_lyr
    )

    # --------------------------------------------------------
    # 7. Select buildings whose center is in blocks_5
    # --------------------------------------------------------

    log(
        "Selecting Overture buildings using "
        "HAVE_THEIR_CENTER_IN..."
    )

    arcpy.management.SelectLayerByLocation(
        in_layer=buildings_lyr,
        overlap_type="HAVE_THEIR_CENTER_IN",
        select_features=blocks_lyr,
        selection_type="NEW_SELECTION"
    )

    selected_count = int(
        arcpy.management.GetCount(buildings_lyr)[0]
    )

    log(f"Buildings selected: {selected_count:,}")

    if selected_count == 0:
        raise RuntimeError(
            "The spatial selection returned zero buildings. "
            "No output will be created."
        )

    # --------------------------------------------------------
    # 8. Export selected building polygons
    # --------------------------------------------------------

    log("")
    log("Exporting selected buildings...")

    arcpy.conversion.ExportFeatures(
        in_features=buildings_lyr,
        out_features=selected_buildings
    )

    exported_count = int(
        arcpy.management.GetCount(selected_buildings)[0]
    )

    log(f"Created:\n  {selected_buildings}")
    log(f"Exported feature count: {exported_count:,}")

    # --------------------------------------------------------
    # 9. Project to UTM Zone 35S
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("PROJECTING BUILDINGS TO UTM 35S")
    log("=" * 80)

    log(f"Target CRS: {target_sr.name}")
    log(f"Target WKID: {target_sr.factoryCode}")

    arcpy.management.Project(
        in_dataset=selected_buildings,
        out_dataset=projected_buildings,
        out_coor_system=target_sr
    )

    projected_count = int(
        arcpy.management.GetCount(projected_buildings)[0]
    )

    projected_sr = arcpy.Describe(
        projected_buildings
    ).spatialReference

    log(f"Created:\n  {projected_buildings}")
    log(f"Projected feature count: {projected_count:,}")
    log(f"Output CRS: {projected_sr.name}")

    # --------------------------------------------------------
    # 10. Clean up temporary feature layers
    # --------------------------------------------------------

    arcpy.management.Delete(buildings_lyr)
    arcpy.management.Delete(blocks_lyr)

    # --------------------------------------------------------
    # Final report
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("DONE")
    log("=" * 80)

    log(f"Input Overture buildings: {building_count:,}")
    log(f"Selected buildings:       {selected_count:,}")
    log(f"Projected buildings:      {projected_count:,}")

    log("")
    log("Outputs:")
    log(f"  {selected_buildings}")
    log(f"  {projected_buildings}")


if __name__ == "__main__":
    try:
        main()
    except Exception:
        log("")
        log("ERROR:")
        log(traceback.format_exc())

In [ ]:
import arcpy
import os
import traceback

# ------------------------------------------------------------
# Input
# ------------------------------------------------------------

blocks_fc = (
    r"E:\_johannesburg\_analysis"
    r"\segments_v2"
    r"\segments.gdb"
    r"\blocks_5"
)

composite_class_field = "composite_class"

# Fields to create
open_space_field = "open_space"
airport_field = "airport"
area_field = "block_area_m2"
perimeter_field = "block_perimeter_m"

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def log(msg):
    print(msg, flush=True)


def get_field(fc, field_name):
    """
    Returns the actual ArcGIS field object using
    case-insensitive matching.
    """
    for field in arcpy.ListFields(fc):
        if field.name.lower() == field_name.lower():
            return field

    return None


def add_field_if_missing(fc, field_name, field_type):
    """
    Add field if missing.

    If it already exists, verify that its type is appropriate.
    """

    existing = get_field(fc, field_name)

    expected_types = {
        "SHORT": "SmallInteger",
        "DOUBLE": "Double"
    }

    if existing is None:
        arcpy.management.AddField(
            in_table=fc,
            field_name=field_name,
            field_type=field_type
        )

        log(
            f"Added field: {field_name} "
            f"({field_type})"
        )

    else:
        expected = expected_types[field_type]

        if existing.type != expected:
            raise TypeError(
                f"Field '{field_name}' already exists, "
                f"but its type is {existing.type}; "
                f"expected {expected}."
            )

        log(
            f"Field already exists: {existing.name} "
            f"({existing.type})"
        )


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():

    log("=" * 80)
    log("CHECKING BLOCKS_5")
    log("=" * 80)

    if not arcpy.Exists(blocks_fc):
        raise FileNotFoundError(
            f"blocks_5 does not exist:\n{blocks_fc}"
        )

    # --------------------------------------------------------
    # Check projected CRS
    # --------------------------------------------------------

    desc = arcpy.Describe(blocks_fc)
    sr = desc.spatialReference

    if sr.type != "Projected":
        raise ValueError(
            "blocks_5 must have a projected coordinate system "
            "before calculating area and perimeter.\n"
            f"Current CRS: {sr.name}"
        )

    log(f"Input: {blocks_fc}")
    log(f"CRS: {sr.name}")

    # --------------------------------------------------------
    # Check composite_class
    # --------------------------------------------------------

    composite_field_obj = get_field(
        blocks_fc,
        composite_class_field
    )

    if composite_field_obj is None:
        raise ValueError(
            f"Required field does not exist: "
            f"{composite_class_field}"
        )

    # Use actual capitalization
    composite_field_actual = composite_field_obj.name

    log(
        f"Classification field: "
        f"{composite_field_actual}"
    )

    # --------------------------------------------------------
    # Add fields
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("ADDING OUTPUT FIELDS")
    log("=" * 80)

    add_field_if_missing(
        blocks_fc,
        open_space_field,
        "SHORT"
    )

    add_field_if_missing(
        blocks_fc,
        airport_field,
        "SHORT"
    )

    add_field_if_missing(
        blocks_fc,
        area_field,
        "DOUBLE"
    )

    add_field_if_missing(
        blocks_fc,
        perimeter_field,
        "DOUBLE"
    )

    # --------------------------------------------------------
    # Make temporary blocks layer
    # --------------------------------------------------------

    blocks_lyr = "blocks_5_calc_lyr"

    if arcpy.Exists(blocks_lyr):
        arcpy.management.Delete(blocks_lyr)

    arcpy.management.MakeFeatureLayer(
        blocks_fc,
        blocks_lyr
    )

    # --------------------------------------------------------
    # OPEN SPACE
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CALCULATING OPEN_SPACE")
    log("=" * 80)

    # First give every record 0
    arcpy.management.CalculateField(
        in_table=blocks_lyr,
        field=open_space_field,
        expression="0",
        expression_type="PYTHON3"
    )

    # Select composite_class = 'open_space'
    composite_sql_field = arcpy.AddFieldDelimiters(
        blocks_fc,
        composite_field_actual
    )

    open_space_where = (
        f"{composite_sql_field} = 'open_space'"
    )

    arcpy.management.SelectLayerByAttribute(
        in_layer_or_view=blocks_lyr,
        selection_type="NEW_SELECTION",
        where_clause=open_space_where
    )

    n_open_space = int(
        arcpy.management.GetCount(blocks_lyr)[0]
    )

    log(f"Open-space blocks selected: {n_open_space:,}")

    # Set selected records to 1
    arcpy.management.CalculateField(
        in_table=blocks_lyr,
        field=open_space_field,
        expression="1",
        expression_type="PYTHON3"
    )

    # Clear selection
    arcpy.management.SelectLayerByAttribute(
        blocks_lyr,
        "CLEAR_SELECTION"
    )

    # --------------------------------------------------------
    # AIRPORT
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CALCULATING AIRPORT")
    log("=" * 80)

    # First give every record 0
    arcpy.management.CalculateField(
        in_table=blocks_lyr,
        field=airport_field,
        expression="0",
        expression_type="PYTHON3"
    )

    airport_where = (
        f"{composite_sql_field} = 'airport'"
    )

    arcpy.management.SelectLayerByAttribute(
        in_layer_or_view=blocks_lyr,
        selection_type="NEW_SELECTION",
        where_clause=airport_where
    )

    n_airport = int(
        arcpy.management.GetCount(blocks_lyr)[0]
    )

    log(f"Airport blocks selected: {n_airport:,}")

    # Set selected records to 1
    arcpy.management.CalculateField(
        in_table=blocks_lyr,
        field=airport_field,
        expression="1",
        expression_type="PYTHON3"
    )

    # Clear selection
    arcpy.management.SelectLayerByAttribute(
        blocks_lyr,
        "CLEAR_SELECTION"
    )

    # --------------------------------------------------------
    # AREA + PERIMETER
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CALCULATING BLOCK GEOMETRY")
    log("=" * 80)

    arcpy.management.CalculateGeometryAttributes(
        in_features=blocks_fc,
        geometry_property=[
            [area_field, "AREA"],
            [perimeter_field, "PERIMETER_LENGTH"]
        ],
        length_unit="METERS",
        area_unit="SQUARE_METERS",
        coordinate_system=sr
    )

    log(
        f"Calculated {area_field} in square meters."
    )

    log(
        f"Calculated {perimeter_field} in meters."
    )

    # --------------------------------------------------------
    # Check results
    # --------------------------------------------------------

    total_blocks = int(
        arcpy.management.GetCount(blocks_fc)[0]
    )

    log("")
    log("=" * 80)
    log("DONE")
    log("=" * 80)

    log(f"Total blocks:      {total_blocks:,}")
    log(f"Open-space blocks: {n_open_space:,}")
    log(f"Airport blocks:    {n_airport:,}")
    log("")
    log("Fields calculated:")
    log(f"  {open_space_field}")
    log(f"  {airport_field}")
    log(f"  {area_field}")
    log(f"  {perimeter_field}")

    # Clean up
    arcpy.management.Delete(blocks_lyr)


if __name__ == "__main__":
    try:
        main()
    except Exception:
        log("")
        log("ERROR:")
        log(traceback.format_exc())

In [ ]:
import arcpy
import os
import traceback

# ------------------------------------------------------------
# Input
# ------------------------------------------------------------

in_fc = (
    r"E:\_johannesburg\_analysis"
    r"\segments_v2"
    r"\segments.gdb"
    r"\blocks_5"
)

# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

out_folder = (
    r"E:\_johannesburg\_analysis"
    r"\segments_v2"
)

out_gpkg = os.path.join(
    out_folder,
    "blocks_5.gpkg"
)

out_layer_name = "blocks_5"

out_fc = os.path.join(
    out_gpkg,
    out_layer_name
)

# ------------------------------------------------------------
# Fields to retain
# ------------------------------------------------------------

keep_fields_requested = [
    "block_id",
    "initial_zones_1_sj",
    "cluster",
    "tie",
    "geoboundaries",
    "population",
    "cluster_revised",
    "concat",
    "zones_5_id",
    "zones_5_pop",
    "open_space",
    "airport",
    "block_area_m2",
    "block_perimeter_m"
]

overwrite_output = True

arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def log(msg):
    print(msg, flush=True)


def field_lookup_case_insensitive(fc):
    """
    Return dictionary:

        lowercase field name -> actual field name

    This accommodates differences such as:

        zones_5_ID
        zones_5_id
    """

    return {
        f.name.lower(): f.name
        for f in arcpy.ListFields(fc)
        if f.type not in [
            "OID",
            "Geometry"
        ]
    }


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

def main():

    # --------------------------------------------------------
    # Check input
    # --------------------------------------------------------

    log("=" * 80)
    log("CHECKING INPUT")
    log("=" * 80)

    if not arcpy.Exists(in_fc):

        raise FileNotFoundError(
            f"Input feature class "
            f"does not exist:\n"
            f"{in_fc}"
        )

    if not os.path.isdir(out_folder):

        raise FileNotFoundError(
            f"Output folder does "
            f"not exist:\n"
            f"{out_folder}"
        )

    input_count = int(
        arcpy.management.GetCount(
            in_fc
        )[0]
    )

    log(
        f"Input:\n"
        f"  {in_fc}"
    )

    log(
        f"Input feature count: "
        f"{input_count:,}"
    )

    # --------------------------------------------------------
    # Resolve requested fields
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CHECKING REQUIRED FIELDS")
    log("=" * 80)

    lookup = field_lookup_case_insensitive(
        in_fc
    )

    keep_fields_actual = []
    missing_fields = []

    for field_name in keep_fields_requested:

        key = field_name.lower()

        if key in lookup:

            keep_fields_actual.append(
                lookup[key]
            )

        else:

            missing_fields.append(
                field_name
            )

    if missing_fields:

        raise ValueError(
            "These required fields were not "
            "found in blocks_5:\n"
            +
            "\n".join(
                f"  - {f}"
                for f in missing_fields
            )
            +
            "\n\nRun the earlier notebook cells "
            "that create/calculate these fields "
            "before exporting."
        )

    log(
        "All required fields are present."
    )

    log("")
    log("Fields to export:")

    for f in keep_fields_actual:

        log(
            f"  {f}"
        )

    # --------------------------------------------------------
    # Delete existing GeoPackage
    # --------------------------------------------------------

    if arcpy.Exists(out_gpkg):

        if overwrite_output:

            log("")
            log(
                f"Deleting existing "
                f"GeoPackage:\n"
                f"  {out_gpkg}"
            )

            arcpy.management.Delete(
                out_gpkg
            )

        else:

            raise FileExistsError(
                f"Output GeoPackage "
                f"already exists:\n"
                f"{out_gpkg}"
            )

    # --------------------------------------------------------
    # Create GeoPackage
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("CREATING GEOPACKAGE")
    log("=" * 80)

    log(
        f"Creating:\n"
        f"  {out_gpkg}"
    )

    arcpy.management.CreateSQLiteDatabase(
        out_gpkg,
        "GEOPACKAGE"
    )

    # --------------------------------------------------------
    # Build field mappings
    # --------------------------------------------------------

    log("")
    log(
        "Building field mappings..."
    )

    field_mappings = (
        arcpy.FieldMappings()
    )

    for field_name in keep_fields_actual:

        fm = arcpy.FieldMap()

        fm.addInputField(
            in_fc,
            field_name
        )

        out_field = fm.outputField

        out_field.name = field_name
        out_field.aliasName = field_name

        fm.outputField = out_field

        field_mappings.addFieldMap(
            fm
        )

    # --------------------------------------------------------
    # Export
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("EXPORTING BLOCKS_5")
    log("=" * 80)

    arcpy.conversion.ExportFeatures(
        in_features=in_fc,
        out_features=out_fc,
        where_clause=None,
        use_field_alias_as_name="NOT_USE_ALIAS",
        field_mapping=field_mappings
    )

    output_count = int(
        arcpy.management.GetCount(
            out_fc
        )[0]
    )

    # --------------------------------------------------------
    # Validate feature count
    # --------------------------------------------------------

    if output_count != input_count:

        log("")
        log("WARNING:")

        log(
            f"  Feature count changed "
            f"during export:"
        )

        log(
            f"  Input:  {input_count:,}"
        )

        log(
            f"  Output: {output_count:,}"
        )

    # --------------------------------------------------------
    # Report
    # --------------------------------------------------------

    log("")
    log("=" * 80)
    log("DONE")
    log("=" * 80)

    log(
        f"Output GeoPackage:\n"
        f"  {out_gpkg}"
    )

    log(
        f"Output layer: "
        f"{out_layer_name}"
    )

    log(
        f"Feature count: "
        f"{output_count:,}"
    )


if __name__ == "__main__":

    try:
        main()

    except Exception:

        log("")
        log("ERROR:")

        log(
            traceback.format_exc()
        )